# 处理数据
- 将GPT评估结果附加到.tsv文件中

In [67]:
import json
import pandas as pd
import os

In [69]:
df = pd.read_csv("uncertainty_scores4.tsv", sep='\t', dtype={'id': str})

with open("intermediate_data_naive.json", "r", encoding="utf-8") as f:
    json_data = json.load(f)

df.head(10)


,id,uncertainty_score
0,687cecd1-d839-4eb1-bf38-638cf5daa579,0.066406
1,8309a492-853c-4873-ae71-08c778825e11,0.117188
2,3c0bacd2-cb3a-40d2-8504-76480582e2bd,0.066406
3,7759e81e-db7a-4774-8040-1150a9475786,0.070312
4,03292bfb-c8b7-46c0-9e32-8dadf12a5509,0.058594
5,a6d2363d-c098-4b35-8a23-4c15c3bba8c0,0.085938
6,0f546023-6988-4737-8fa5-8519e92969af,0.070312
7,fee9400b-708f-454d-a6ab-f311d40824da,0.082031
8,25923538-8c50-4af4-a28f-3f0a76c8ae9d,0.074219
9,804ff703-e647-4c50-93ed-14de080fabc8,0.062500


In [70]:
id_to_score_map = {}
for item in json_data:
    item_id = str(item.get('id'))
    score = item['output']['metric_score']['gpt']
    id_to_score_map[item_id] = score

In [71]:
new_col_name = 'gpt_score'
df[new_col_name] = df['id'].map(id_to_score_map)
missing_count = df[new_col_name].isna().sum()
if missing_count > 0:
    print(f"注意: 有 {missing_count} 行数据在 TSV 中存在但在 JSON 中未找到对应的 ID (填充为空)。")

In [72]:
df.to_csv("uncertainty_scores4_with_gpt.tsv", sep='\t', index=False)

# 计算Pearson

In [73]:
import numpy as np
from sklearn.metrics import roc_auc_score

In [78]:
df = pd.read_csv('uncertainty_scores4_with_gpt.tsv', sep='\t')
df = df[df['uncertainty_score'] != 0.0]
len(df)

600

In [79]:
# 确保数据是数值类型
df['uncertainty_score'] = pd.to_numeric(df['uncertainty_score'], errors='coerce')
df['gpt_score'] = pd.to_numeric(df['gpt_score'], errors='coerce')
df = df.dropna() # 再次移除转换数字失败的行

In [80]:
pearson_corr = df['uncertainty_score'].corr(df['gpt_score'], method='pearson')
pearson_corr

-0.019343312299835835

# 计算AUROC

In [81]:
y_true = df['gpt_score'].values
y_score = df['uncertainty_score']
auroc = roc_auc_score(y_true, y_score)
auroc

0.5042201834862385